# Separate T/N classification and multitask pain regression

This notebook documents an exploratory research iteration using questionnaire scores, V/P/K-coded features and tumour-site indicators. It is part of the [experiment series](../docs/experiments.md).

**Recorded outputs:** the aggregate results below were saved in the original research notebook and retained after privacy review. They have not been recomputed during presentation cleanup. Row previews, per-record target arrays, raw-data exports, local warnings and repetitive epoch logs were removed. Execution counts were cleared because the original execution order was incomplete.

**Re-running:** launch Jupyter from the repository root, install the research dependencies, and supply an authorized final masterchart CSV. Run cells in order. Do not commit newly generated outputs before reviewing them. See [data setup](../data/README.md).

**Interpretation:** these are within-cohort experiments, not a validated clinical model or proof of a semantic relationship. Numeric conversion fills unavailable feature values with zero. Historical pain scaling and site encoding use the complete filtered cohort; fold-local preprocessing and independent validation remain follow-up work.


## Environment and imports


In [ ]:
import numpy as np  
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix



## Portable local input


In [ ]:
from pathlib import Path
import os

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
DATA_PATH = Path(os.environ.get("MEDTECH_FINAL_DATA_PATH", REPO_ROOT / "data" / "medtech_final.csv"))
if not DATA_PATH.is_file():
    raise FileNotFoundError("Supply the authorized final masterchart CSV at data/medtech_final.csv or set MEDTECH_FINAL_DATA_PATH.")
df = pd.read_csv(DATA_PATH)


## Cohort dimensions


In [ ]:
df.shape

(226, 329)


## Schema overview


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226 entries, 0 to 225
Columns: 329 entries, S.no to K_Q67
dtypes: float64(54), int64(273), object(2)
memory usage: 581.0+ KB


## Missing-value inspection


In [ ]:
print(df.isnull().sum().sort_values(ascending=False).head(30))


P_Q16          141
P_Q19          105
V_Q9            61
P_Q9            61
V_Q18           55
K_Q4            11
V_Q4            11
P_Q13           10
K_Q13           10
P_Q56.3          0
P_Q59            0
P_Q59_final      0
P_Q58            0
P_Q57            0
P_Q56.4          0
P_Q56            0
P_Q56.2          0
P_Q56.1          0
P_Q59.2          0
P_Q56_final      0
P_Q55            0
P_Q54            0
P_Q53.1          0
P_Q53            0
P_Q59.1          0
P_Q62_final      0
P_Q60            0
P_Q61            0
K_Q3             0
K_Q2.1           0
dtype: int64


## Select final questionnaire columns


In [ ]:
# Get all column names
cols = df.columns

# Columns to keep
keep_cols = []

for col in cols:
    # If it's a "_final" column → always keep
    if col.endswith('_final'):
        keep_cols.append(col)
    else:
        # Check if corresponding _final exists
        base = col.split('.')[0]  # handles K_Q65.1
        final_col = base + '_final'
        
        if final_col not in cols:
            keep_cols.append(col)

# Keep only selected columns
df = df[keep_cols]

In [ ]:
df.shape

(226, 251)


In [ ]:
print(df.columns.tolist())

['S.no', 'Age', 'weight', 'height', 'Pulse rate', 'Pain', 'Site', 'staging (TNM)', 'Vata', 'Pitta', 'Kapha', 'SF 36', 'Value_Q1', 'Value_Q2', 'Value_Q3', 'Value_Q4', 'Value_Q5', 'Value_Q6', 'Value_Q7', 'Value_Q8', 'Value_Q9', 'Value_Q10', 'Value_Q11', 'Value_Q12', 'Value_Q13', 'Value_Q14', 'Value_Q15', 'Value_Q16', 'Value_Q17', 'Value_Q18', 'Value_Q19', 'Value_Q20', 'Value_Q21', 'Value_Q22', 'Value_Q23', 'Value_Q24', 'Value_Q25', 'Value_Q26', 'Value_Q27', 'Value_Q28', 'Value_Q29', 'Value_Q30', 'Value_Q31', 'Value_Q32', 'Value_Q33', 'Value_Q34', 'Value_Q35', 'Value_Q36', 'V_Q1', 'V_Q2_final', 'V_Q3', 'V_Q4', 'V_Q5', 'V_Q6', 'V_Q7', 'V_Q8', 'V_Q9', 'V_Q10', 'V_Q11', 'V_Q12', 'V_Q13', 'V_Q14', 'V_Q15', 'V_Q16', 'V_Q17_final', 'V_Q18', 'V_Q19', 'V_Q20', 'V_Q21_final', 'V_Q22', 'V_Q23', 'V_Q24', 'V_Q25', 'V_Q26', 'V_Q27', 'V_Q28', 'V_Q29', 'V_Q30', 'V_Q31', 'V_Q32', 'V_Q33', 'V_Q34', 'V_Q35', 'V_Q36', 'V_Q37', 'V_Q38', 'V_Q39', 'V_Q40', 'V_Q41', 'V_Q42', 'V_Q43', 'V_Q44', 'V_Q45', 'V_Q46', 

In [ ]:
for col in df.columns:
    print(f'{col}: {df[col].dtypes}')

S.no: int64
Age: int64
weight: int64
height: int64
Pulse rate: int64
Pain: int64
Site: object
staging (TNM): object
Vata: float64
Pitta: float64
Kapha: float64
SF 36: int64
Value_Q1: int64
Value_Q2: int64
Value_Q3: int64
Value_Q4: int64
Value_Q5: int64
Value_Q6: int64
Value_Q7: int64
Value_Q8: int64
Value_Q9: int64
Value_Q10: int64
Value_Q11: int64
Value_Q12: int64
Value_Q13: int64
Value_Q14: int64
Value_Q15: int64
Value_Q16: int64
Value_Q17: int64
Value_Q18: int64
Value_Q19: int64
Value_Q20: int64
Value_Q21: float64
Value_Q22: int64
Value_Q23: float64
Value_Q24: float64
Value_Q25: float64
Value_Q26: float64
Value_Q27: float64
Value_Q28: float64
Value_Q29: float64
Value_Q30: float64
Value_Q31: float64
Value_Q32: int64
Value_Q33: int64
Value_Q34: int64
Value_Q35: int64
Value_Q36: int64
V_Q1: int64
V_Q2_final: int64
V_Q3: int64
V_Q4: float64
V_Q5: int64
V_Q6: int64
V_Q7: int64
V_Q8: int64
V_Q9: float64
V_Q10: int64
V_Q11: int64
V_Q12: int64
V_Q13: int64
V_Q14: int64
V_Q15: int64
V_Q16: i

In [ ]:
df['Site'].value_counts()

Site
Oral cavity                 62
Supraglottic                39
Glottic                     33
Hypopharyngeal              30
Oropharyngeal               23
Sinonasal carcinoma         10
Nasopharyngeal Carcinoma     8
Maxillary sinus              5
Sinonasal Carcinoma          4
Glottis                      3
Esophagus                    2
Esophageal                   1
Sinonasal chondrosarcoma     1
Oropharynx                   1
sinonasal carcinoma          1
Sinonasal Sarcoma            1
Nasopharyngeal carcinoma     1
Transglottic                 1
Name: count, dtype: int64


## Parse separate T and N targets


In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

def split_tnm_into_t_n(raw, keep_suffixes=False):
    """
    Extracts T and N from a raw TNM string.

    Examples:
        T2aN1bM0  ->  T2, N1   (default)
        T4bN2cM1  ->  T4, N2   (default)
        TisN0M0   ->  Tis, N0

    If keep_suffixes=True:
        T2a -> T2a
        N1b -> N1b
    """
    if pd.isna(raw):
        return np.nan, np.nan

    s = str(raw).upper().strip()
    s = re.sub(r'[\s\-_\/]+', '', s)
    s = re.sub(r'[^TNM0-9ABCIS]', '', s)

    t_match = re.search(r'T(IS|[0-4][ABC]?)', s)
    n_match = re.search(r'N([0-3][ABC]?)', s)

    if not (t_match and n_match):
        return np.nan, np.nan

    t_tok = t_match.group(1)
    n_tok = n_match.group(1)

    # T label
    if t_tok == "IS":
        t_label = "Tis"
    else:
        t_base = t_tok[0]
        t_suffix = t_tok[1:] if len(t_tok) > 1 else ""
        t_label = f"T{t_base}{t_suffix.lower()}" if keep_suffixes and t_suffix else f"T{t_base}"

    # N label
    n_base = n_tok[0]
    n_suffix = n_tok[1:] if len(n_tok) > 1 else ""
    n_label = f"N{n_base}{n_suffix.lower()}" if keep_suffixes and n_suffix else f"N{n_base}"

    return t_label, n_label


def add_tn_columns(df, stage_col="staging (TNM)", keep_suffixes=False):
    """
    Adds:
      - T_stage : encoded T label
      - N_stage : encoded N label

    Also returns the fitted encoders.
    """
    df = df.copy()

    if stage_col not in df.columns:
        raise ValueError(f"Column '{stage_col}' not found")

    parsed = df[stage_col].apply(lambda x: split_tnm_into_t_n(x, keep_suffixes=keep_suffixes))
    parsed_df = pd.DataFrame(parsed.tolist(), columns=["T_label", "N_label"], index=df.index)

    df = pd.concat([df, parsed_df], axis=1)

    # keep only rows where both T and N were successfully parsed
    mask = df["T_label"].notna() & df["N_label"].notna()
    df = df.loc[mask].copy()

    t_le = LabelEncoder()
    n_le = LabelEncoder()

    df["T_stage"] = t_le.fit_transform(df["T_label"].astype(str))
    df["N_stage"] = n_le.fit_transform(df["N_label"].astype(str))

    # remove helper string columns
    df.drop(columns=["T_label", "N_label"], inplace=True)

    return df, t_le, n_le


# Usage
df, t_le, n_le = add_tn_columns(df, stage_col="staging (TNM)", keep_suffixes=False)

print("T classes:", list(t_le.classes_))
print("N classes:", list(n_le.classes_))


T classes: ['T1', 'T2', 'T3', 'T4']
N classes: ['N0', 'N1', 'N2', 'N3']


## Historical numeric filtering and scaling

This preserves the historical complete-cohort min-max scaling, including Pain. For confirmatory evaluation, fit preprocessing on training folds and specify the outcome scale independently.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# -----------------------------
# 1. Define numeric columns
# -----------------------------
numeric_cols = ["Pain", "weight", "height", "Pulse rate"]

# Ensure numeric conversion
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

# -----------------------------
# 2. Filter valid rows
# -----------------------------
# Now we use T_stage and N_stage instead of stage_clean
required_cols = ["T_stage", "N_stage"] + numeric_cols

mask = pd.Series(True, index=df.index)

for col in required_cols:
    mask &= df[col].notna()

df = df.loc[mask].copy()

print("After filtering:", df.shape)

# -----------------------------
# 3. Min-Max scaling (replace original columns)
# -----------------------------
scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# -----------------------------
# Final dataset ready
# -----------------------------
print("\nFinal shape:", df.shape)


Final shape: (226, 253)


## Encode tumour-site categories


In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# ensure df is DataFrame
print(type(df))

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded = encoder.fit_transform(df[["Site"]])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(["Site"]), index=df.index
)

df = pd.concat([df, encoded_df], axis=1)
df.drop("Site", axis=1, inplace=True)

## Build VALUE, VPK and ALL feature settings


In [ ]:
import pandas as pd
import numpy as np
import re

def prepare_dataset(df, mode="value"):
    """
    mode options:
    - "value" → Value_Q1 to Value_Q36
    - "vpk"   → V_Q*, P_Q*, K_Q*
    - "all"   → Value_Q* + V_Q* + P_Q* + K_Q*

    Returns:
      X        : feature matrix
      y_T      : T_stage
      y_N      : N_stage
      y_pain   : Pain
    """

    df = df.copy()

    # -----------------------------
    # 1. Clean column names
    # -----------------------------
    df.columns = df.columns.str.strip()
    df.columns = [re.sub(r'\.\d+$', '', col) for col in df.columns]
    df.columns = [col.replace('-', '_') for col in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]

    # -----------------------------
    # 2. Targets
    # -----------------------------
    required_targets = ["T_stage", "N_stage", "Pain"]

    for col in required_targets:
        if col not in df.columns:
            raise ValueError(f"Target column '{col}' not found")

    y_T = pd.to_numeric(df["T_stage"], errors="coerce")
    y_N = pd.to_numeric(df["N_stage"], errors="coerce")
    y_pain = pd.to_numeric(df["Pain"], errors="coerce")

    # Keep only valid rows
    mask = y_T.notna() & y_N.notna() & y_pain.notna()
    df = df.loc[mask].copy()

    y_T = y_T.loc[mask].astype(int)
    y_N = y_N.loc[mask].astype(int)
    y_pain = y_pain.loc[mask].astype(float)

    # -----------------------------
    # 3. Select feature columns
    # -----------------------------
    if mode == "value":
        feature_cols = [col for col in df.columns if col.startswith("Value_Q")]

    elif mode == "vpk":
        feature_cols = [
            col for col in df.columns
            if col.startswith("V_Q") or col.startswith("P_Q") or col.startswith("K_Q")
        ]

    elif mode == "all":
        feature_cols = [
            col for col in df.columns
            if col.startswith("Value_Q")
            or col.startswith("V_Q")
            or col.startswith("P_Q")
            or col.startswith("K_Q")
        ]

    else:
        raise ValueError("Invalid mode. Choose from: value, vpk, all")

    # -----------------------------
    # 4. Add Site encoded columns
    # -----------------------------
    site_cols = [col for col in df.columns if col.startswith("Site_")]

    # -----------------------------
    # 5. Final feature matrix
    # -----------------------------
    final_cols = feature_cols + site_cols

    X = (
        df[final_cols]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    return (
        np.nan_to_num(np.asarray(X, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0),
        np.asarray(y_T),
        np.asarray(y_N),
        np.asarray(y_pain),
    )


# -----------------------------
# Example usage
# -----------------------------
X1, y_T1, y_N1, y_pain1 = prepare_dataset(df, mode="value")
X2, y_T2, y_N2, y_pain2 = prepare_dataset(df, mode="vpk")
X3, y_T3, y_N3, y_pain3 = prepare_dataset(df, mode="all")

print(X1.shape, X2.shape, X3.shape)
print("T classes:", len(np.unique(y_T1)))
print("N classes:", len(np.unique(y_N1)))
print("Pain range:", y_pain1.min(), y_pain1.max())

(226, 54) (226, 220) (226, 256)
T classes: 4
N classes: 4
Pain range: 0.0 1.0


## Rare-class handling


In [ ]:
import numpy as np
from collections import Counter

def fix_single_sample_classes(X, y_T, y_N, y_pain, min_count=3):
    """
    Ensures each (T, N) class combination has at least `min_count` samples
    by duplicating rows.

    Parameters:
        X       : feature matrix (numpy array)
        y_T     : T_stage labels
        y_N     : N_stage labels
        y_pain  : regression target
        min_count : minimum samples required per class

    Returns:
        X_new, y_T_new, y_N_new, y_pain_new
    """

    X = np.asarray(X)
    y_T = np.asarray(y_T)
    y_N = np.asarray(y_N)
    y_pain = np.asarray(y_pain)

    # 🔥 Combine T and N into joint class
    y_joint = np.array([f"{t}_{n}" for t, n in zip(y_T, y_N)])

    class_counts = Counter(y_joint)

    X_list = [X]
    yT_list = [y_T]
    yN_list = [y_N]
    ypain_list = [y_pain]

    for cls, count in class_counts.items():
        if count < min_count:
            needed = min_count - count

            # indices of this joint class
            idx = np.where(y_joint == cls)[0]

            # duplicate
            dup_idx = np.random.choice(idx, size=needed, replace=True)

            X_list.append(X[dup_idx])
            yT_list.append(y_T[dup_idx])
            yN_list.append(y_N[dup_idx])
            ypain_list.append(y_pain[dup_idx])

    # combine
    X_new = np.vstack(X_list)
    y_T_new = np.concatenate(yT_list)
    y_N_new = np.concatenate(yN_list)
    y_pain_new = np.concatenate(ypain_list)

    return X_new, y_T_new, y_N_new, y_pain_new

## Multitask architecture definitions


In [ ]:
import numpy as np
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


class MultiTaskNet(nn.Module):
    def __init__(
        self,
        input_dim,
        n_T_classes,
        n_N_classes,
        hidden_dims=(64, 32, 16),
        dropout=0.10
    ):
        super().__init__()

        h1, h2, h3 = hidden_dims

        # -----------------------------
        # Shared backbone
        # -----------------------------
        self.shared = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h2, h3),
            nn.ReLU()
        )

        # -----------------------------
        # T classification head
        # -----------------------------
        self.T_head = nn.Sequential(
            nn.Linear(h3, h3),
            nn.ReLU(),
            nn.Linear(h3, n_T_classes)
        )

        # -----------------------------
        # N classification head
        # -----------------------------
        self.N_head = nn.Sequential(
            nn.Linear(h3, h3),
            nn.ReLU(),
            nn.Linear(h3, n_N_classes)
        )

        # -----------------------------
        # Pain regression head
        # -----------------------------
        self.pain_head = nn.Sequential(
            nn.Linear(h3, h3),
            nn.ReLU(),
            nn.Linear(h3, 1)
        )

    def forward(self, x):
        shared = self.shared(x)

        T_logits = self.T_head(shared)
        N_logits = self.N_head(shared)
        pain_pred = self.pain_head(shared)

        return T_logits, N_logits, pain_pred


# ---------------------------------------------------
# Build function with 3 architectures
# ---------------------------------------------------
def build_model(input_dim, n_T_classes, n_N_classes, arch="big"):
    """
    arch options:
    - "small" → 32-16-8
    - "big"   → 64-32-16
    - "ultra" → 128-64-32
    """

    if arch == "small":
        hidden_dims = (32, 16, 8)
        dropout = 0.05

    elif arch == "big":
        hidden_dims = (64, 32, 16)
        dropout = 0.10

    elif arch == "ultra":
        hidden_dims = (128, 64, 32)
        dropout = 0.15  # slightly higher to control overfitting

    else:
        raise ValueError("arch must be 'small', 'big', or 'ultra'")

    model = MultiTaskNet(
        input_dim=input_dim,
        n_T_classes=n_T_classes,
        n_N_classes=n_N_classes,
        hidden_dims=hidden_dims,
        dropout=dropout
    )

    return model.to(device)

Using device: cpu


## Aggregate diagnostic metrics


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    explained_variance_score,
    confusion_matrix
)
import numpy as np


# ---------------------------------------------------
# Regression metrics (UNCHANGED)
# ---------------------------------------------------
def regression_debug_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)

    pred_var = float(np.var(y_pred))
    true_var = float(np.var(y_true))
    resid = y_true - y_pred
    resid_std = float(np.std(resid))

    return {
        "MAE": float(mae),
        "MSE": float(mse),
        "RMSE": rmse,
        "R2": float(r2),
        "ExplainedVariance": float(evs),
        "PredVariance": pred_var,
        "TrueVariance": true_var,
        "ResidualStd": resid_std,
    }


# ---------------------------------------------------
# Classification metrics (for ONE head)
# ---------------------------------------------------
def classification_debug_metrics(y_true, y_pred, y_prob=None):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    out = {
        "Accuracy": float(acc),
        "BalancedAccuracy": float(bal_acc),
        "MacroF1": float(macro_f1),
        "ConfusionMatrix": confusion_matrix(y_true, y_pred)
    }

    if y_prob is not None:
        y_prob = np.asarray(y_prob)
        max_conf = np.max(y_prob, axis=1)
        entropy = -np.sum(y_prob * np.log(np.clip(y_prob, 1e-12, 1.0)), axis=1)

        out["MeanSoftmaxConfidence"] = float(np.mean(max_conf))
        out["MedianSoftmaxConfidence"] = float(np.median(max_conf))
        out["MeanEntropy"] = float(np.mean(entropy))

    return out


# ---------------------------------------------------
# Print block for BOTH T and N
# ---------------------------------------------------
def print_metrics_block(title, T_metrics, N_metrics, reg_metrics):
    print(f"\n===== {title} =====")

    # -----------------------------
    # T metrics
    # -----------------------------
    print("\n--- T Classification ---")
    print(
        f"Acc: {T_metrics['Accuracy']:.4f} | "
        f"BalAcc: {T_metrics['BalancedAccuracy']:.4f} | "
        f"MacroF1: {T_metrics['MacroF1']:.4f}"
    )

    if "MeanSoftmaxConfidence" in T_metrics:
        print(
            f"Confidence | Mean: {T_metrics['MeanSoftmaxConfidence']:.4f} | "
            f"Median: {T_metrics['MedianSoftmaxConfidence']:.4f} | "
            f"Entropy: {T_metrics['MeanEntropy']:.4f}"
        )

    print("Confusion matrix (T):\n", T_metrics["ConfusionMatrix"])

    # -----------------------------
    # N metrics
    # -----------------------------
    print("\n--- N Classification ---")
    print(
        f"Acc: {N_metrics['Accuracy']:.4f} | "
        f"BalAcc: {N_metrics['BalancedAccuracy']:.4f} | "
        f"MacroF1: {N_metrics['MacroF1']:.4f}"
    )

    if "MeanSoftmaxConfidence" in N_metrics:
        print(
            f"Confidence | Mean: {N_metrics['MeanSoftmaxConfidence']:.4f} | "
            f"Median: {N_metrics['MedianSoftmaxConfidence']:.4f} | "
            f"Entropy: {N_metrics['MeanEntropy']:.4f}"
        )

    print("Confusion matrix (N):\n", N_metrics["ConfusionMatrix"])

    # -----------------------------
    # Pain regression
    # -----------------------------
    print("\n--- Pain Regression ---")
    print(
        f"MAE: {reg_metrics['MAE']:.4f} | "
        f"RMSE: {reg_metrics['RMSE']:.4f} | "
        f"MSE: {reg_metrics['MSE']:.4f} | "
        f"R2: {reg_metrics['R2']:.4f} | "
        f"EVS: {reg_metrics['ExplainedVariance']:.4f}"
    )

    print(
        f"Variance | Pred: {reg_metrics['PredVariance']:.4f} | "
        f"True: {reg_metrics['TrueVariance']:.4f} | "
        f"ResidualStd: {reg_metrics['ResidualStd']:.4f}"
    )

## Training and evaluation runner


In [ ]:
from collections import Counter
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

def run_model(
    X,
    y_T,
    y_N,
    y_pain,
    dataset_name="dataset",
    arch="big",
    min_count=3,
    test_size=0.20,
    random_state=42,
    batch_size=16,
    epochs=120,
    lr=3e-3,
    weight_decay=1e-5,
    T_loss_w=1.0,
    N_loss_w=1.0,
    pain_loss_w=0.5
):

    # ---------------------------------------------------
    # 1) Convert to numpy
    # ---------------------------------------------------
    X = np.asarray(X, dtype=np.float32)
    y_T = np.asarray(y_T, dtype=np.int64).reshape(-1)
    y_N = np.asarray(y_N, dtype=np.int64).reshape(-1)
    y_pain = np.asarray(y_pain, dtype=np.float32).reshape(-1, 1)

    print(f"\n--- {dataset_name.upper()} ---")
    print("Original shape:", X.shape)
    print("T class counts:", dict(Counter(y_T)))
    print("N class counts:", dict(Counter(y_N)))

    # ---------------------------------------------------
    # 2) Train/test split (FIRST)
    # ---------------------------------------------------
    X_train, X_test, yT_train, yT_test, yN_train, yN_test, ypain_train, ypain_test = train_test_split(
        X,
        y_T,
        y_N,
        y_pain,
        test_size=test_size,
        random_state=random_state,
        stratify=y_T   # stratify on T (more stable than joint)
    )

    # ---------------------------------------------------
    # 3) Apply replication ONLY on train
    # ---------------------------------------------------
    X_train, yT_train, yN_train, ypain_train = fix_single_sample_classes(
        X_train, yT_train, yN_train, ypain_train, min_count=min_count
    )

    print("Train shape (after balancing):", X_train.shape)
    print("Test shape:", X_test.shape)

    # ---------------------------------------------------
    # 4) DataLoaders
    # ---------------------------------------------------
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    yT_train_t = torch.tensor(yT_train, dtype=torch.long)
    yN_train_t = torch.tensor(yN_train, dtype=torch.long)
    ypain_train_t = torch.tensor(ypain_train, dtype=torch.float32)

    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    yT_test_t = torch.tensor(yT_test, dtype=torch.long)
    yN_test_t = torch.tensor(yN_test, dtype=torch.long)
    ypain_test_t = torch.tensor(ypain_test, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, yT_train_t, yN_train_t, ypain_train_t)
    test_ds = TensorDataset(X_test_t, yT_test_t, yN_test_t, ypain_test_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    # ---------------------------------------------------
    # 5) Model
    # ---------------------------------------------------
    n_T_classes = len(np.unique(y_T))
    n_N_classes = len(np.unique(y_N))

    model = build_model(
        input_dim=X.shape[1],
        n_T_classes=n_T_classes,
        n_N_classes=n_N_classes,
        arch=arch
    )

    T_criterion = nn.CrossEntropyLoss()
    N_criterion = nn.CrossEntropyLoss()
    pain_criterion = nn.MSELoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    total_steps = max(1, epochs * len(train_loader))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr,
        total_steps=total_steps,
        pct_start=0.25,
        anneal_strategy="cos",
        div_factor=10.0,
        final_div_factor=100.0
    )

    use_amp = device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    # ---------------------------------------------------
    # 6) Training loop
    # ---------------------------------------------------
    for epoch in range(1, epochs + 1):
        model.train()

        total_loss = 0
        T_correct = 0
        N_correct = 0
        total = 0

        for xb, yTb, yNb, ypb in train_loader:
            xb = xb.to(device)
            yTb = yTb.to(device)
            yNb = yNb.to(device)
            ypb = ypb.to(device).squeeze(-1)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                T_logits, N_logits, pain_pred = model(xb)
                pain_pred = pain_pred.squeeze(-1)

                loss_T = T_criterion(T_logits, yTb)
                loss_N = N_criterion(N_logits, yNb)
                loss_pain = pain_criterion(pain_pred, ypb)

                loss = (
                    T_loss_w * loss_T +
                    N_loss_w * loss_N +
                    pain_loss_w * loss_pain
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item() * xb.size(0)

            T_pred = torch.argmax(T_logits, dim=1)
            N_pred = torch.argmax(N_logits, dim=1)

            T_correct += (T_pred == yTb).sum().item()
            N_correct += (N_pred == yNb).sum().item()
            total += yTb.size(0)

        # print(
        #     f"Epoch {epoch:03d} | "
        #     f"Loss: {total_loss/total:.4f} | "
        #     f"T_acc: {T_correct/total:.4f} | "
        #     f"N_acc: {N_correct/total:.4f}"
        # )

    # ---------------------------------------------------
    # 7) Evaluation
    # ---------------------------------------------------
    model.eval()

    all_T_true, all_T_pred, all_T_prob = [], [], []
    all_N_true, all_N_pred, all_N_prob = [], [], []
    all_pain_true, all_pain_pred = [], []

    with torch.no_grad():
        for xb, yTb, yNb, ypb in test_loader:
            xb = xb.to(device)

            T_logits, N_logits, pain_pred = model(xb)

            T_prob = torch.softmax(T_logits, dim=1)
            N_prob = torch.softmax(N_logits, dim=1)

            all_T_true.extend(yTb.numpy())
            all_T_pred.extend(torch.argmax(T_prob, dim=1).cpu().numpy())
            all_T_prob.extend(T_prob.cpu().numpy())

            all_N_true.extend(yNb.numpy())
            all_N_pred.extend(torch.argmax(N_prob, dim=1).cpu().numpy())
            all_N_prob.extend(N_prob.cpu().numpy())

            all_pain_true.extend(ypb.numpy())
            all_pain_pred.extend(pain_pred.squeeze(-1).cpu().numpy())

    T_metrics = classification_debug_metrics(
        all_T_true, all_T_pred, all_T_prob
    )
    N_metrics = classification_debug_metrics(
        all_N_true, all_N_pred, all_N_prob
    )
    reg_metrics = regression_debug_metrics(
        all_pain_true, all_pain_pred
    )

    print_metrics_block(dataset_name, T_metrics, N_metrics, reg_metrics)

    return {
        "model": model,
        "T_metrics": T_metrics,
        "N_metrics": N_metrics,
        "regression": reg_metrics
    }

## Nine feature-set / architecture comparisons


In [ ]:

# -----------------------------
# VALUE MODE
# -----------------------------
results_value_small = run_model(
    X1, y_T1, y_N1, y_pain1,
    dataset_name="value_small",
    arch="small"
)

results_value_big = run_model(
    X1, y_T1, y_N1, y_pain1,
    dataset_name="value_big",
    arch="big"
)

results_value_ultra = run_model(
    X1, y_T1, y_N1, y_pain1,
    dataset_name="value_ultra",
    arch="ultra"
)


# -----------------------------
# VPK MODE
# -----------------------------
results_vpk_small = run_model(
    X2, y_T2, y_N2, y_pain2,
    dataset_name="vpk_small",
    arch="small"
)

results_vpk_big = run_model(
    X2, y_T2, y_N2, y_pain2,
    dataset_name="vpk_big",
    arch="big"
)

results_vpk_ultra = run_model(
    X2, y_T2, y_N2, y_pain2,
    dataset_name="vpk_ultra",
    arch="ultra"
)


# -----------------------------
# ALL MODE
# -----------------------------
results_all_small = run_model(
    X3, y_T3, y_N3, y_pain3,
    dataset_name="all_small",
    arch="small"
)

results_all_big = run_model(
    X3, y_T3, y_N3, y_pain3,
    dataset_name="all_big",
    arch="big"
)

results_all_ultra = run_model(
    X3, y_T3, y_N3, y_pain3,
    dataset_name="all_ultra",
    arch="ultra"
)


--- VALUE_SMALL ---
Original shape: (226, 54)
T class counts: {1: 35, 2: 133, 3: 53, 0: 5}
N class counts: {2: 48, 1: 106, 0: 58, 3: 14}
Train shape (after balancing): (182, 54)
Test shape: (46, 54)



===== value_small =====

--- T Classification ---
Acc: 0.5435 | BalAcc: 0.2449 | MacroF1: 0.2098
Confidence | Mean: 0.6480 | Median: 0.6084 | Entropy: 0.8223
Confusion matrix (T):
 [[ 0  0  1  0]
 [ 0  0  6  1]
 [ 0  1 24  2]
 [ 0  0 10  1]]

--- N Classification ---
Acc: 0.3913 | BalAcc: 0.2829 | MacroF1: 0.2515
Confidence | Mean: 0.6005 | Median: 0.5245 | Entropy: 0.9073
Confusion matrix (N):
 [[ 5  6  0  0]
 [ 6 12  2  0]
 [ 3  9  1  0]
 [ 2  0  0  0]]

--- Pain Regression ---
MAE: 0.1202 | RMSE: 0.1565 | MSE: 0.0245 | R2: 0.4718 | EVS: 0.4846
Variance | Pred: 0.0123 | True: 0.0464 | ResidualStd: 0.1546

--- VALUE_BIG ---
Original shape: (226, 54)
T class counts: {1: 35, 2: 133, 3: 53, 0: 5}
N class counts: {2: 48, 1: 106, 0: 58, 3: 14}
Train shape (after balancing): (182, 54)
Test shape: (46, 54)



===== value_big =====

--- T Classification ---
Acc: 0.4348 | BalAcc: 0.2386 | MacroF1: 0.2458
Confidence | Mean: 0.6907 | Median: 0.6518 | Entropy: 0.7090
Confusion matrix (T):
 [[ 0  0  0  1]
 [ 0  1  5  1]
 [ 3  4 17  3]
 [ 1  2  6  2]]

--- N Classification ---
Acc: 0.3696 | BalAcc: 0.2704 | MacroF1: 0.2506
Confidence | Mean: 0.6610 | Median: 0.6259 | Entropy: 0.7954
Confusion matrix (N):
 [[ 5  3  3  0]
 [ 4 11  5  0]
 [ 3  9  1  0]
 [ 2  0  0  0]]

--- Pain Regression ---
MAE: 0.1280 | RMSE: 0.1656 | MSE: 0.0274 | R2: 0.4080 | EVS: 0.4146
Variance | Pred: 0.0145 | True: 0.0464 | ResidualStd: 0.1647

--- VALUE_ULTRA ---
Original shape: (226, 54)
T class counts: {1: 35, 2: 133, 3: 53, 0: 5}
N class counts: {2: 48, 1: 106, 0: 58, 3: 14}
Train shape (after balancing): (182, 54)
Test shape: (46, 54)



===== value_ultra =====

--- T Classification ---
Acc: 0.4783 | BalAcc: 0.2706 | MacroF1: 0.2753
Confidence | Mean: 0.7747 | Median: 0.8291 | Entropy: 0.5204
Confusion matrix (T):
 [[ 0  0  0  1]
 [ 0  1  6  0]
 [ 2  3 18  4]
 [ 0  2  6  3]]

--- N Classification ---
Acc: 0.3261 | BalAcc: 0.2489 | MacroF1: 0.1990
Confidence | Mean: 0.7349 | Median: 0.6866 | Entropy: 0.6251
Confusion matrix (N):
 [[6 5 0 0]
 [9 9 2 0]
 [4 9 0 0]
 [1 1 0 0]]

--- Pain Regression ---
MAE: 0.1101 | RMSE: 0.1450 | MSE: 0.0210 | R2: 0.5466 | EVS: 0.5506
Variance | Pred: 0.0195 | True: 0.0464 | ResidualStd: 0.1443

--- VPK_SMALL ---
Original shape: (226, 220)
T class counts: {1: 35, 2: 133, 3: 53, 0: 5}
N class counts: {2: 48, 1: 106, 0: 58, 3: 14}
Train shape (after balancing): (182, 220)
Test shape: (46, 220)



===== vpk_small =====

--- T Classification ---
Acc: 0.6087 | BalAcc: 0.3535 | MacroF1: 0.3208
Confidence | Mean: 0.7078 | Median: 0.7114 | Entropy: 0.6864
Confusion matrix (T):
 [[ 0  0  1  0]
 [ 0  0  7  0]
 [ 0  0 21  6]
 [ 0  0  4  7]]

--- N Classification ---
Acc: 0.3913 | BalAcc: 0.2759 | MacroF1: 0.2777
Confidence | Mean: 0.6419 | Median: 0.6276 | Entropy: 0.8758
Confusion matrix (N):
 [[ 3  6  0  2]
 [ 3 12  4  1]
 [ 3  7  3  0]
 [ 0  2  0  0]]

--- Pain Regression ---
MAE: 0.1714 | RMSE: 0.2272 | MSE: 0.0516 | R2: -0.1134 | EVS: -0.1118
Variance | Pred: 0.0162 | True: 0.0464 | ResidualStd: 0.2270

--- VPK_BIG ---
Original shape: (226, 220)
T class counts: {1: 35, 2: 133, 3: 53, 0: 5}
N class counts: {2: 48, 1: 106, 0: 58, 3: 14}
Train shape (after balancing): (182, 220)
Test shape: (46, 220)



===== vpk_big =====

--- T Classification ---
Acc: 0.5217 | BalAcc: 0.2761 | MacroF1: 0.2647
Confidence | Mean: 0.7722 | Median: 0.8230 | Entropy: 0.5172
Confusion matrix (T):
 [[ 0  0  1  0]
 [ 0  0  7  0]
 [ 0  1 20  6]
 [ 0  3  4  4]]

--- N Classification ---
Acc: 0.4348 | BalAcc: 0.3415 | MacroF1: 0.3538
Confidence | Mean: 0.7720 | Median: 0.8059 | Entropy: 0.5143
Confusion matrix (N):
 [[5 5 0 1]
 [3 9 5 3]
 [2 5 6 0]
 [0 2 0 0]]

--- Pain Regression ---
MAE: 0.1599 | RMSE: 0.2157 | MSE: 0.0465 | R2: -0.0037 | EVS: 0.0088
Variance | Pred: 0.0078 | True: 0.0464 | ResidualStd: 0.2143

--- VPK_ULTRA ---
Original shape: (226, 220)
T class counts: {1: 35, 2: 133, 3: 53, 0: 5}
N class counts: {2: 48, 1: 106, 0: 58, 3: 14}
Train shape (after balancing): (182, 220)
Test shape: (46, 220)



===== vpk_ultra =====

--- T Classification ---
Acc: 0.5652 | BalAcc: 0.3345 | MacroF1: 0.3307
Confidence | Mean: 0.8004 | Median: 0.8464 | Entropy: 0.4613
Confusion matrix (T):
 [[ 0  0  1  0]
 [ 0  1  6  0]
 [ 0  1 20  6]
 [ 0  3  3  5]]

--- N Classification ---
Acc: 0.5217 | BalAcc: 0.5108 | MacroF1: 0.4960
Confidence | Mean: 0.8251 | Median: 0.8490 | Entropy: 0.4537
Confusion matrix (N):
 [[ 5  5  1  0]
 [ 4 11  4  1]
 [ 0  5  7  1]
 [ 0  1  0  1]]

--- Pain Regression ---
MAE: 0.1731 | RMSE: 0.2275 | MSE: 0.0518 | R2: -0.1171 | EVS: -0.1128
Variance | Pred: 0.0130 | True: 0.0464 | ResidualStd: 0.2271

--- ALL_SMALL ---
Original shape: (226, 256)
T class counts: {1: 35, 2: 133, 3: 53, 0: 5}
N class counts: {2: 48, 1: 106, 0: 58, 3: 14}
Train shape (after balancing): (182, 256)
Test shape: (46, 256)



===== all_small =====

--- T Classification ---
Acc: 0.5435 | BalAcc: 0.2854 | MacroF1: 0.2646
Confidence | Mean: 0.7901 | Median: 0.8997 | Entropy: 0.5055
Confusion matrix (T):
 [[ 0  0  1  0]
 [ 0  0  7  0]
 [ 0  0 21  6]
 [ 0  1  6  4]]

--- N Classification ---
Acc: 0.3478 | BalAcc: 0.2307 | MacroF1: 0.1956
Confidence | Mean: 0.6595 | Median: 0.5934 | Entropy: 0.7765
Confusion matrix (N):
 [[ 3  7  1  0]
 [ 4 13  3  0]
 [ 4  9  0  0]
 [ 0  2  0  0]]

--- Pain Regression ---
MAE: 0.1386 | RMSE: 0.1720 | MSE: 0.0296 | R2: 0.3614 | EVS: 0.3638
Variance | Pred: 0.0218 | True: 0.0464 | ResidualStd: 0.1717

--- ALL_BIG ---
Original shape: (226, 256)
T class counts: {1: 35, 2: 133, 3: 53, 0: 5}
N class counts: {2: 48, 1: 106, 0: 58, 3: 14}
Train shape (after balancing): (182, 256)
Test shape: (46, 256)



===== all_big =====

--- T Classification ---
Acc: 0.5435 | BalAcc: 0.3123 | MacroF1: 0.2971
Confidence | Mean: 0.8161 | Median: 0.8865 | Entropy: 0.4164
Confusion matrix (T):
 [[ 0  0  1  0]
 [ 0  0  7  0]
 [ 0  2 19  6]
 [ 0  2  3  6]]

--- N Classification ---
Acc: 0.4130 | BalAcc: 0.3088 | MacroF1: 0.3109
Confidence | Mean: 0.7849 | Median: 0.8381 | Entropy: 0.5385
Confusion matrix (N):
 [[ 5  5  0  1]
 [ 3 11  4  2]
 [ 3  7  3  0]
 [ 0  2  0  0]]

--- Pain Regression ---
MAE: 0.1160 | RMSE: 0.1432 | MSE: 0.0205 | R2: 0.5574 | EVS: 0.5592
Variance | Pred: 0.0178 | True: 0.0464 | ResidualStd: 0.1429

--- ALL_ULTRA ---
Original shape: (226, 256)
T class counts: {1: 35, 2: 133, 3: 53, 0: 5}
N class counts: {2: 48, 1: 106, 0: 58, 3: 14}
Train shape (after balancing): (182, 256)
Test shape: (46, 256)



===== all_ultra =====

--- T Classification ---
Acc: 0.5870 | BalAcc: 0.3303 | MacroF1: 0.3340
Confidence | Mean: 0.8653 | Median: 0.9542 | Entropy: 0.3191
Confusion matrix (T):
 [[ 0  0  1  0]
 [ 0  1  6  0]
 [ 0  2 22  3]
 [ 0  1  6  4]]

--- N Classification ---
Acc: 0.3261 | BalAcc: 0.2419 | MacroF1: 0.2356
Confidence | Mean: 0.8756 | Median: 0.9660 | Entropy: 0.3301
Confusion matrix (N):
 [[4 6 0 1]
 [7 9 3 1]
 [3 8 2 0]
 [0 2 0 0]]

--- Pain Regression ---
MAE: 0.1305 | RMSE: 0.1627 | MSE: 0.0265 | R2: 0.4287 | EVS: 0.4453
Variance | Pred: 0.0151 | True: 0.0464 | ResidualStd: 0.1603
